# Studio di MERGE
basato su cleaning 4
## Inizializzazione ed Import

In [1]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from data_model.manage_excel_support_file import *
from data_model.MergerTools import *
import pandas as pd
import os

client = DatalakeClient()
mergeTools = MergerTools()

# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake


**Mixed info**\
'ADNIMERGE', \
'ADSP_PHC_BIOMARKER', 'ADNI_DIAN_COMPARISON', --> have CSF

**Single Cofactor**\
'PTDEMOG', 'DXSUM', 'MMSE', 'ADAS', 'FAQ', 'CDR', 'MOCA', 'APOERES'

**Volumes**\
'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSL', 'UCSDVOL', 'UPENN_ROI_MARS',\
'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSX51_ADNI1_3T'  --> just partial immages segmentation

**CSF**\
'UPENNBIOMK_ADNIDIAN_ES_2017', 'UPENNBIOMK_ROCHE_ELECSYS', 'EUROIMMUN', 'FUJIREBIOABETA', 'SALADAX_BIOMEDICAL', 'MESOSCALE', 'UPENNBIOMK_MASTER', 'UPENN_2DUPLC_CRM', 

In [2]:
file_codes =['ADNIMERGE', 'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'MMSE',
                'UCSFFSL', 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSX51_ADNI1_3T']
search = client.query_files(
    query={'custom.level' : 'cleaned_04', 'custom.source' : 'ADNI', 'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

print(len(zip_files))

11


In [3]:
##### MODIFICARE 
category = 'volumes'

In [6]:
dfs = {}
df_names = {}
df_code = []
idx = 0
for file_name, df_raw in zip_files.items():
    print('\n### ', file_name)
    df_copy = df_raw.copy(deep=True)
    len_pre = len(df_copy.columns)
    row_pre = len(df_copy)
    # get the sub_df focusing on the category chosen
    df_copy = mergeTools.filter_df_category(df_copy, category)
    len_post = len(df_copy.columns)
    row_post = len(df_copy)
    if df_copy.empty:
        print(file_name, f'------> non ha colonne dellca categoria {category}')
        continue
    # get the common columns among all the dfs
    if idx == 0:
        dfs_columns = set(df_copy.columns)
    else: 
        dfs_columns &= set(df_copy.columns)
    # Ensure EXAMDATE in date format and correct order of the dfs
    df_copy['EXAMDATE'] = pd.to_datetime(df_copy['EXAMDATE'])
    df_copy = df_copy.sort_values(by=['RID', 'EXAMDATE']).reset_index(drop=True)
    if 'FSVERSION' in df_copy.columns:
        df_copy['FSVERSION'] = df_copy['FSVERSION'].astype(str)
    # aggiornamento liste e dizionari
    dfs[f"df_{idx}"] = df_copy  
    df_names[f"df_{idx}"] = file_name 
    df_code.append(f"df_{idx}")
    # definizione variabile df
    globals()[f"df_{idx}"] = df_copy
    print(idx, '--->', file_name, '\t\t\t### ', len_post, '/', len_pre, '\n\t\t\t\t\t\t rows: ', row_post, '/', row_pre)
    idx += 1

time_buffer = pd.Timedelta(days=80)


###  ADNIMERGE_25Jul2025_04.csv
0 ---> ADNIMERGE_25Jul2025_04.csv 			###  14 / 41 
						 rows:  9321 / 11458

###  MMSE_25Jul2025_04.csv
MMSE_25Jul2025_04.csv ------> non ha colonne dellca categoria volumes

###  UCSFFSX_11_02_15_11Aug2025_04.csv
1 ---> UCSFFSX_11_02_15_11Aug2025_04.csv 			###  14 / 14 
						 rows:  4076 / 4076

###  UCSFFSX7_11Aug2025_04.csv
2 ---> UCSFFSX7_11Aug2025_04.csv 			###  15 / 15 
						 rows:  847 / 847

###  UCSFFSX6_11Aug2025_04.csv
3 ---> UCSFFSX6_11Aug2025_04.csv 			###  14 / 14 
						 rows:  2222 / 2222

###  UCSFFSX51_11_08_19_11Aug2025_04.csv
4 ---> UCSFFSX51_11_08_19_11Aug2025_04.csv 			###  14 / 14 
						 rows:  4038 / 4038

###  UCSFFSX51_ADNI1_3T_02_01_16_11Aug2025_04.csv
5 ---> UCSFFSX51_ADNI1_3T_02_01_16_11Aug2025_04.csv 			###  14 / 14 
						 rows:  435 / 435

###  UCSFFSL51ALL_08_01_16_11Aug2025_04.csv
6 ---> UCSFFSL51ALL_08_01_16_11Aug2025_04.csv 			###  14 / 14 
						 rows:  424 / 424

###  UCSFFSL51_03_01_22_11Aug2025_04.csv
7 ---> 

In [5]:
x = 0
for df_x in dfs.values():
    print(x, df_x['FSVERSION'].unique())
    x += 1

0 ['4.3' '5.1' '6.0']
1 ['4.3']
2 ['7.4.1']
3 ['6.0']
4 ['5.1']
5 ['5.1']
6 ['5.1']
7 ['5.1']
8 ['5.1']
9 ['4.4']


# Confronto stessi RID  ==> RID - EXAMDATE identici tra file

In [ ]:

subj_matrix = mergeTools.matrix_match(dfs, df_names, df_code, columns_list=['RID'])
print("righe con stessi ####### RID:")
display(subj_matrix)
        
subj_date_matrix = mergeTools.matrix_match(dfs, df_names, df_code, columns_list=['RID', 'EXAMDATE'], time_buffer=time_buffer)
print("righe con stessi ####### RID-EXAMDATE: --> time_buffer=", time_buffer)
display(subj_date_matrix)

if set(['FSVERSION', 'IMAGEUID']).issubset(set(dfs_columns)):
    subj_viscode_matrix = mergeTools.matrix_match(dfs, df_names, df_code, columns_list=['RID', 'EXAMDATE', 'FSVERSION','IMAGEUID'])
    print("righe con stessi ####### RID-EXAMDATE-FSVERSION-IMAGEUID: --> time_buffer=", time_buffer)
    display(subj_viscode_matrix)

if set('METHOD').issubset(set(dfs_columns)):
    subj_viscode_matrix = mergeTools.matrix_match(dfs, df_names, df_code, columns_list=['RID', 'EXAMDATE', 'METHOD'])
    print("righe con stessi ####### RID-EXAMDATE-METHOD: --> time_buffer=", time_buffer)
    display(subj_viscode_matrix)

righe con stessi ####### RID:


,df_0,df_1,df_2,df_3,df_4,df_5,df_6,df_7,df_8,df_9
df_0,2350,,,,,,,,,
df_1,818,842,,,,,,,,
df_2,263,11,807,,,,,,,
df_3,968,64,281,1122,,,,,,
df_4,980,78,71,313,1049,,,,,
df_5,138,138,3,12,13,138,,,,
df_6,113,1,1,12,113,1,113,,,
df_7,662,25,48,209,662,5,89,662,,
df_8,347,7,25,122,347,2,30,292,347,
df_9,739,739,11,62,77,132,1,25,7,739


righe con stessi ####### RID-EXAMDATE: --> time_buffer= 80 days 00:00:00


,df_0,df_1,df_2,df_3,df_4,df_5,df_6,df_7,df_8,df_9
df_0,9321,,,,,,,,,
df_1,3863,4076,,,,,,,,
df_2,0,0,847,,,,,,,
df_3,1761,0,0,2222,,,,,,
df_4,3631,5,0,0,4038,,,,,
df_5,414,411,0,0,0,435,,,,
df_6,380,0,0,0,401,0,424,,,
df_7,2859,0,0,0,2965,0,357,3123,,
df_8,1158,1,0,0,1259,0,100,1071,1283,
df_9,3317,3333,0,0,3,396,0,0,0,3354


righe con stessi ####### RID-EXAMDATE-FSVERSION-IMAGEUID: --> time_buffer= 80 days 00:00:00


,df_0,df_1,df_2,df_3,df_4,df_5,df_6,df_7,df_8,df_9
df_0,9321,,,,,,,,,
df_1,1508,4076,,,,,,,,
df_2,0,0,847,,,,,,,
df_3,536,0,0,2222,,,,,,
df_4,1501,0,0,0,4038,,,,,
df_5,0,0,0,0,0,435,,,,
df_6,161,0,0,0,388,0,424,,,
df_7,1235,0,0,0,2812,0,357,3123,,
df_8,461,0,0,0,1166,0,100,1071,1283,
df_9,0,0,0,0,0,0,0,0,0,3354


# Inizio Merge
## Definizione di df_base e Gerarchia di DF da mergiare
Scegliere il file con numero maggiore di soggetti-visite e che ha più elementi con altri df.\
Quindi scegliere con che ordine unire gli altri df, suggerimento da quelli con nessuna/pochissime righe RID-EXAMDATE in comune con gli altri df, e quindi quelli con molte righe in comune a partire da quello con più righe in comune sia con df_base che con gli altri e quindi a seguire. Ma di persè il metodo è arbitrario quindi si può fare come si vuole.

In [ ]:
############   MODIFICARE
base = 'df_4'
df_base = dfs[base].copy(deep=True)                           #sembra un errore ma questi df sono definiti
idx_add = [ 'df_6', 'df_7', 'df_8', 'df_9','df_1', 'df_2', 'df_3', 'df_5', 'df_0']
merge_contains = [base]
sub_with_match = set()
time_buffer = pd.Timedelta(days=80)
i = 0

## Approfondimento RID-EXAMDATE
1. vedo quante righe ci sono con match ESATTO e quante con TIME BUFFER.

In [97]:
df_add = dfs[idx_add[i]].copy(deep=True)
print(idx_add[i])

df_0


In [98]:
exact_matches, buffer_matches = mergeTools.find_visit_matches(df_base, df_add, buffer_days=time_buffer)
exact_index1, exact_index2 = mergeTools.list_index_visit_matches(exact_matches)
buff_index1, buff_index2 = mergeTools.list_index_visit_matches(buffer_matches)


print('Exact matches: \t', len(exact_index1))
print('Buffered matches: \t', len(buff_index1))

if len(buff_index1) != len(buff_index2):
    print('\n ------>>> ATTENZIONE: indici match con buffer SPAIATI')

if len(buff_index1) != len(buff_index2):
    print('\n ------>>> ATTENZIONE: indici match SPAIATI')

Exact matches: 	 5232
Buffered matches: 	 8104


In [99]:
columns_in_common = list(df_base.columns.intersection(df_add.columns))
columns_only_base = list(df_base.columns.difference(df_add.columns))
columns_only_add = list(df_add.columns.difference(df_base.columns))

all_index1 = exact_index1.union(buff_index1)
all_index2 = exact_index2.union(buff_index2)

# studio le colonne in comune e non ai due df
print('Le colonne in comune sono:\n', columns_in_common)
print('\n\nLe colonne solo in df_base sono: \n', columns_only_base)
print('\n\nLe colonne solo in df_add sono: \n', columns_only_add)
print('__________________________________________________________________\n\n')

diff_date = False
# verifico ci siano righe matchate con BUFFER
if len(buff_index1) == len(buff_index2) and len(buff_index1) > 0:    
    diff_date = True
    # verifico se le righe matchate sono TUTTE matchate con BUFFER
    if all(all_index1) == len(all_index2) and all_index1 == buff_index1 and all_index2 == buff_index2:
        print(f'all maches have a buffer, tot: {len(all_index1)} matches\n\n====================================> Renamen\'EXAMDATE\' column\n')
    else:
        print(f'There are {len(buff_index1)} match with buffer\nThere are {len(all_index1)-len(buff_index1)} match exact\nOver {len(all_index1)} total matches\n\n====================================> SHOULD \'EXAMDATE\' column be renamed?\n')
    

# ci sono solo match ESATTI
elif len(buff_index1) == len(buff_index2) and len(buff_index1) == 0 and len(all_index1) > 0:
    print('JUST exact matches')
elif len(buff_index1) == len(buff_index2) and len(buff_index1) == 0 and len(all_index1) == 0:
    print('NO matches')


Le colonne in comune sono:
 ['COHORT', 'RID', 'VISCODE', 'VISIT_MONTH', 'EXAMDATE', 'IMAGEUID', 'FSVERSION', 'ICV%ICV', 'Fusiform%ICV', 'Hippocampus%ICV', 'Ventricles%ICV', 'Entorhinal%ICV', 'MidTemp%ICV', 'FLDSTRENG']


Le colonne solo in df_base sono: 
 ['STATUS']


Le colonne solo in df_add sono: 
 []
__________________________________________________________________


There are 8104 match with buffer
There are 5232 match exact
Over 13336 total matches

====================================> SHOULD 'EXAMDATE' column be renamed?



In [100]:
if diff_date:
    print('df_add --> ', df_names[idx_add[i]], '\ndf_base --> ', [df_names[x] for x in merge_contains])
    col_list = list(df_base.columns) + [c for c in df_add.columns if c not in df_base.columns]
    
    temp_merge = mergeTools.create_temp_merge(df_base, df_add, buff_index1, buff_index2, col_list=col_list)
    diff = temp_merge['EXAMDATE_1']-temp_merge['EXAMDATE_2']
    display(diff[diff != pd.Timedelta(days=0)])
    print(len(diff[diff != pd.Timedelta(days=0)]))
    display(temp_merge.loc[diff[diff != pd.Timedelta(days=0)].index])
    

df_add -->  ADNIMERGE_25Jul2025_04.csv 
df_base -->  ['UCSFFSX51_11_08_19_11Aug2025_04.csv', 'UCSFFSL51ALL_08_01_16_11Aug2025_04.csv', 'UCSFFSL51_03_01_22_11Aug2025_04.csv', 'UCSFFSL51Y1_08_01_16_11Aug2025_04.csv', 'UCSFFSL_02_01_16_11Aug2025_04.csv', 'UCSFFSX_11_02_15_11Aug2025_04.csv', 'UCSFFSX7_11Aug2025_04.csv', 'UCSFFSX6_11Aug2025_04.csv', 'UCSFFSX51_ADNI1_3T_02_01_16_11Aug2025_04.csv']


0      -13 days
1      -11 days
2      -11 days
3      -47 days
4      -47 days
         ...   
8099   -10 days
8100   -17 days
8101   -13 days
8102    -7 days
8103   -26 days
Length: 8104, dtype: timedelta64[ns]

8104


,COHORT_1,COHORT_2,RID,VISCODE_1,VISCODE_2,VISIT_MONTH_1,VISIT_MONTH_2,EXAMDATE_1,EXAMDATE_2,IMAGEUID_1,...,Hippocampus%ICV_1,Hippocampus%ICV_2,Ventricles%ICV_1,Ventricles%ICV_2,Entorhinal%ICV_1,Entorhinal%ICV_2,MidTemp%ICV_1,MidTemp%ICV_2,FLDSTRENG_1,FLDSTRENG_2
0,NaN,ADNI1,2,sc,bl,0,0,2005-08-26,2005-09-08,35475,...,0.420022,0.420022,5.702085,5.957343,0.210464,0.210464,1.407596,1.407596,1.5T,1.5T
1,NaN,ADNI1,3,sc,bl,0,0,2005-09-01,2005-09-12,32237,...,0.276932,0.276932,3.768333,4.404615,0.093248,0.093248,0.959134,0.959134,1.5T,1.5T
2,NaN,ADNI1,3,sc,bl,0,0,2005-09-01,2005-09-12,32237,...,0.270528,0.276932,3.737928,4.404615,0.122404,0.093248,0.939454,0.959134,1.5T,1.5T
3,NaN,ADNI1,4,sc,bl,0,0,2005-09-22,2005-11-08,64631,...,0.409005,0.409005,2.293681,2.358227,0.237162,0.237162,1.167949,1.167949,1.5T,1.5T
4,NaN,ADNI1,4,sc,bl,0,0,2005-09-22,2005-11-08,64631,...,0.380186,0.409005,2.289870,2.358227,0.231208,0.237162,1.163662,1.167949,1.5T,1.5T
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8099,ADNI3,ADNI3,7085,sc,bl,0,0,2022-07-01,2022-07-11,1600180,...,0.505566,0.505566,1.739894,1.774093,0.272865,0.272865,1.440623,1.440623,NaN,NaN
8100,ADNI3,ADNI3,7088,sc,bl,0,0,2022-06-27,2022-07-14,1598296,...,0.391395,0.391396,3.080373,3.295828,0.271960,0.271960,1.217736,1.217738,NaN,NaN
8101,ADNI3,ADNI3,7092,sc,bl,0,0,2022-06-29,2022-07-12,1597666,...,0.462679,0.462680,4.705639,4.898167,0.268625,0.268625,1.301092,1.301092,NaN,NaN
8102,ADNI3,ADNI3,7100,sc,bl,0,0,2022-09-07,2022-09-14,1619004,...,0.471259,0.471261,1.987607,2.024802,0.232734,0.232735,1.336229,1.336234,NaN,NaN


In [101]:
filtered_merge = temp_merge[temp_merge['VISCODE_1']!=temp_merge['VISCODE_2']]
print('tot:    ', len(filtered_merge))
print('s-bl:   ', len(filtered_merge[(filtered_merge['VISCODE_1']=='sc') & (filtered_merge['VISCODE_2']=='bl')]))
print('f-bl:   ', len(filtered_merge[(filtered_merge['VISCODE_1']=='f') & (filtered_merge['VISCODE_2']=='bl')]))

tot:     3543
s-bl:    2074
f-bl:    6


In [102]:
filtered_merge[~((filtered_merge['VISCODE_1'].isin(['f', 'sc'])) & (filtered_merge['VISCODE_2']=='bl'))]

,COHORT_1,COHORT_2,RID,VISCODE_1,VISCODE_2,VISIT_MONTH_1,VISIT_MONTH_2,EXAMDATE_1,EXAMDATE_2,IMAGEUID_1,...,Hippocampus%ICV_1,Hippocampus%ICV_2,Ventricles%ICV_1,Ventricles%ICV_2,Entorhinal%ICV_1,Entorhinal%ICV_2,MidTemp%ICV_1,MidTemp%ICV_2,FLDSTRENG_1,FLDSTRENG_2
1121,NaN,ADNI1,314,m06,m12,11,12,2007-02-26,2007-05-02,65152,...,0.374946,0.385542,1.918761,2.067639,0.176773,0.140224,1.155912,1.166201,1.5T,1.5T
1122,NaN,ADNI1,314,m06,m12,11,12,2007-02-26,2007-05-02,65152,...,0.345923,0.385542,1.923099,2.067639,0.170910,0.140224,1.173325,1.166201,1.5T,1.5T
1246,NaN,ADNI2,361,bl,m60,61,60,2011-05-20,2011-05-09,297272,...,0.342489,0.356277,2.798375,3.036042,0.131439,0.141872,0.945665,1.007833,1.5T,1.5T
1318,NaN,ADNI2,382,bl,m60,61,60,2011-06-16,2011-06-13,297274,...,0.418874,0.445429,2.043736,2.139139,0.234884,0.240172,1.191604,1.227893,1.5T,1.5T
2115,NaN,ADNI2,618,m60,m72,72,71,2012-06-20,2012-06-13,370058,...,0.543545,0.541126,1.117150,1.113923,0.308755,0.275165,1.242078,1.166371,1.5T,1.5T
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7275,ADNI2,ADNI2,5288,scmri,bl,0,0,2013-09-18,2013-09-25,392161,...,0.493774,0.493774,3.152888,3.373592,0.243106,0.243106,1.471499,1.471499,NaN,3T
7279,ADNI2,ADNI2,5289,scmri,bl,0,0,2013-09-03,2013-11-11,398319,...,0.520089,0.510683,2.312325,2.310390,0.245166,0.240108,1.565247,1.600957,NaN,3T
7280,ADNI2,ADNI2,5290,scmri,bl,0,0,2013-09-06,2013-09-24,391092,...,0.538666,0.538666,0.684029,0.740103,0.283869,0.283869,1.370702,1.370702,NaN,3T
7282,ADNI2,ADNI2,5292,scmri,bl,0,0,2013-10-18,2013-11-12,398341,...,0.555447,0.555447,1.101822,1.163216,0.296337,0.296337,1.389556,1.389556,NaN,3T


In [71]:
modify_examdate = False #True   #False
if modify_examdate:
    df_add.loc[buff_index2, 'EXAMDATE'] = df_base.loc[buff_index1, 'EXAMDATE'].values
    display(df_add.loc[buff_index2]['EXAMDATE'])

In [ ]:
modify_examdate = False #True   #False
index_modified = []
if modify_examdate:
    for x1, x2 in zip(buff_index1, buff_index2):
        if df_base.loc[x1, 'VISCODE'] == df_add.loc[x2, 'VISCODE'] or df_base.loc[x1, 'EXAMDATE'] - df_add.loc[x2, 'EXAMDATE'] < pd.Timedelta(days=30):
            df_add.loc[x2, 'EXAMDATE'] = df_base.loc[x1, 'EXAMDATE']
            index_modified.append([x1, x2])

print(len(index_modified))

7650


## Studio Colonne in comune per righe che matchano
1) Se ci sono righe che machano tra i due df allora identifico altre colonne in comune ai due df.

2) Faccio merge tra i due df escludendo i soggetti che hanno visite metchate tra i 2 df (righe). --> merge_base solo aggiunta di soggetti nuovi.

3) Quindi se ci sono colonne in comune e righe che matchano faccio merge soggetto per soggetto (tra i soggetti con  visite in entrambi i df).\
Aggiungo qusti merge di singoli soggetti al resto del merge_base.


Così ottengo Merge finale.

In [ ]:
df_merged = mergeTools.get_merged_df(df_add, df_base, category=category)

### merging volumes
-----> Merged df (24637 rows)
		-----> duplicates removed (8755 rows)
-----> final df (15882 rows)


In [105]:
df_base = df_merged.copy(deep=True)
merge_contains.append(idx_add[i])
#prepare for next merge
i += 1
print(f'il merg contine i seguenti df: {merge_contains}')
if i <= len(idx_add)-1:
    print(f'il prossimo df da unire è: {idx_add[i]}\n\n ===> torna al capitolo: "Approfondimento RID-EXAMDATE"')
else:
    print('FINISHED MERGE!!!!!')

il merg contine i seguenti df: ['df_4', 'df_6', 'df_7', 'df_8', 'df_9', 'df_1', 'df_2', 'df_3', 'df_5', 'df_0']
FINISHED MERGE!!!!!


# SALVAREEEE

In [ ]:
new_file_name = category + '_merged.csv'
file_code = 'VOLMERGE'

# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=df_merged,
    object_name=new_file_name,
    prefix='cleaned/merged/category',
    metadata={
        'level': 'merged',
        'file_code': file_code,
        'source': 'ADNI'
    }
)